# Sesi 4 — Auto-Report Generation Reference
Notebook referensi untuk trainer. Peserta menulis kode bertahap mengikuti slide.


In [32]:
from app.report_service import generate_report
result = generate_report('2026-05-11', '2026-05-17', 'html')
result


{'status': 'success',
 'report_id': 'weekly_sales_2026-05-11_2026-05-17',
 'markdown': 'C:\\Users\\Rayyanusa\\Downloads\\ai_auto_report_reference_solution\\Pertemuan 5\\outputs\\markdown\\weekly_sales_2026-05-11_2026-05-17.md',
 'html': 'C:\\Users\\Rayyanusa\\Downloads\\ai_auto_report_reference_solution\\Pertemuan 5\\outputs\\html\\weekly_sales_2026-05-11_2026-05-17.html',
 'pdf': None,
 'log': 'C:\\Users\\Rayyanusa\\Downloads\\ai_auto_report_reference_solution\\Pertemuan 5\\outputs\\logs\\weekly_sales_2026-05-11_2026-05-17.json',
 'alert_message': '[AUTO-REPORT ALERT]\n- HIGH | Revenue channel TikTok turun -37.2% vs periode sebelumnya\n  Cek: Cek campaign, diskon, stok, retur, dan perubahan traffic channel.\n- MEDIUM | Revenue channel Shopee turun -20.1% vs periode sebelumnya\n  Cek: Cek campaign, diskon, stok, retur, dan perubahan traffic channel.\nReport lengkap: C:\\Users\\Rayyanusa\\Downloads\\ai_auto_report_reference_solution\\Pertemuan 5\\outputs\\html\\weekly_sales_2026-05-11_2

Buka file HTML/Markdown yang dihasilkan pada folder outputs.


In [33]:
from app.config import DATABASE_URL
from app.data_access import pull_sales_data

df = pull_sales_data(DATABASE_URL, "2026-05-11", "2026-05-17")

print(df.shape)
print(df["order_date"].min(), df["order_date"].max())
print(df["channel"].unique())
df.head()


(487, 13)
2026-05-11 2026-05-17
<StringArray>
['Shopee', 'TikTok', 'Tokopedia', 'Website']
Length: 4, dtype: str


,order_id,order_date,channel,region,sku,product_name,category,qty,unit_price,discount_amount,revenue,cost,status
0,ORD-003531,2026-05-11,Shopee,Kalimantan,ADC,Acne Defense Cream,Treatment,3,145000,43500,391500,192000,completed
1,ORD-003532,2026-05-11,Shopee,Jabodetabek,ADC,Acne Defense Cream,Treatment,1,145000,29000,116000,64000,completed
2,ORD-003528,2026-05-11,Shopee,Jawa Tengah,ECG,Eye Care Gel,Treatment,1,175000,17500,157500,92000,completed
3,ORD-003533,2026-05-11,Shopee,Sumatera,ECG,Eye Care Gel,Treatment,1,175000,8800,166200,92000,completed
4,ORD-003536,2026-05-11,Shopee,Jawa Timur,ECG,Eye Care Gel,Treatment,1,175000,17500,157500,92000,completed


In [34]:
from app.kpi import build_kpi_tables

summary, by_channel, by_sku = build_kpi_tables(df)

print(summary)
print(by_channel)
print(by_sku.head(10))


   total_revenue  total_qty  gross_margin  margin_rate  avg_order_value  \
0     84914370.0        628    39633370.0       0.4667         135214.0   

   row_count  
0        487  
     channel  total_revenue  total_qty  total_cost  order_count  gross_margin  \
0     Shopee       26987170        204    15235000          158      11752170   
1     TikTok       22190020        164    11585000          126      10605020   
2  Tokopedia       20226180        152    10693000          115       9533180   
3    Website       15511000        108     7768000           88       7743000   

   margin_rate  
0       0.4355  
1       0.4779  
2       0.4713  
3       0.4992  
    sku          product_name  total_revenue  total_qty  order_count
4  SSUP      Sunscreen SPF 50       23950080        167          129
3   SER     Brightening Serum       18018000         96           71
0   ADC    Acne Defense Cream       14485080        113           86
1   ECG          Eye Care Gel       11665000        

In [35]:
from datetime import datetime, timedelta

def previous_period(start_date, end_date):
    start = datetime.fromisoformat(start_date)
    end = datetime.fromisoformat(end_date)
    days = (end - start).days + 1
    prev_end = start - timedelta(days=1)
    prev_start = prev_end - timedelta(days=days-1)
    return prev_start.date().isoformat(), prev_end.date().isoformat()

previous_period("2026-05-11", "2026-05-17")


('2026-05-04', '2026-05-10')

In [36]:
def calculate_growth(current, previous):
    cur = current.groupby("channel", as_index=False)         .agg(current_revenue=("revenue", "sum"))
    prev = previous.groupby("channel", as_index=False)         .agg(previous_revenue=("revenue", "sum"))

    growth = cur.merge(prev, on="channel", how="outer").fillna(0)
    growth["growth_rate"] = (
        growth["current_revenue"] - growth["previous_revenue"]
    ) / growth["previous_revenue"].replace(0, 1)

    return growth.sort_values("growth_rate")


In [37]:
current_df = pull_sales_data(DATABASE_URL, "2026-05-11", "2026-05-17")
prev_start, prev_end = previous_period("2026-05-11", "2026-05-17")
previous_df = pull_sales_data(DATABASE_URL, prev_start, prev_end)

growth_by_channel = calculate_growth(current_df, previous_df)

display(growth_by_channel)

,channel,current_revenue,previous_revenue,growth_rate
1,TikTok,22190020,35324380,-0.371821
0,Shopee,26987170,33795820,-0.201464
3,Website,15511000,17956800,-0.136205
2,Tokopedia,20226180,20743910,-0.024958


In [38]:
import pandas as pd

def detect_anomalies(growth_table, threshold=-0.20):
    alerts = []
    for _, row in growth_table.iterrows():
        if row["growth_rate"] < threshold:
            alerts.append({
                "metric": "revenue_growth",
                "dimension": row["channel"],
                "severity": "high" if row["growth_rate"] < -0.30 else "medium",
                "message": f"Revenue channel {row['channel']} turun {row['growth_rate']:.1%}",
                "recommended_check": "Cek campaign, diskon, stok, retur, traffic."
            })
    return pd.DataFrame(alerts)


In [39]:
def build_alert_message(alert_df, report_path):
    if alert_df.empty:
        return "Tidak ada alert prioritas pada periode ini."

    lines = ["[AUTO-REPORT ALERT]"]
    for _, row in alert_df.iterrows():
        lines.append(f"- {row['severity'].upper()} | {row['message']}")
        lines.append(f"  Cek: {row['recommended_check']}")

    lines.append(f"Report lengkap: {report_path}")
    return "\n".join(lines)


In [40]:
from app.alerts import detect_anomalies
alert_df = detect_anomalies(growth_by_channel)
display(alert_df)


,metric,dimension,severity,message,recommended_check
0,revenue_growth,TikTok,high,Revenue channel TikTok turun -37.2% vs periode...,"Cek campaign, diskon, stok, retur, dan perubah..."
1,revenue_growth,Shopee,medium,Revenue channel Shopee turun -20.1% vs periode...,"Cek campaign, diskon, stok, retur, dan perubah..."


In [41]:
message = build_alert_message(
    alert_df,"outputs/pdf/weekly_2026_05_11-2026_05_17.pdf"
)
print(message)

[AUTO-REPORT ALERT]
- HIGH | Revenue channel TikTok turun -37.2% vs periode sebelumnya
  Cek: Cek campaign, diskon, stok, retur, dan perubahan traffic channel.
- MEDIUM | Revenue channel Shopee turun -20.1% vs periode sebelumnya
  Cek: Cek campaign, diskon, stok, retur, dan perubahan traffic channel.
Report lengkap: outputs/pdf/weekly_2026_05_11-2026_05_17.pdf


In [42]:
from pathlib import Path
from jinja2 import Template
import markdown

ROOT_DIR = Path.cwd()
OUTPUT_DIR = ROOT_DIR / "outputs"
HTML_TEMPLATE_PATH = ROOT_DIR / "templates" / "report_template.html"

def export_html(report_md, report_id):
    template = Template(HTML_TEMPLATE_PATH.read_text(encoding="utf-8"))
    html_body = markdown.markdown(report_md, extensions=["tables"])

    html = template.render(
        content=html_body,
        owner="BI Team",
        metadata={
            "start_date": "2026-05-11",
            "end_date": "2026-05-17"
        }
    )

    path = OUTPUT_DIR / "html" / f"{report_id}.html"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(html, encoding="utf-8")
    return path

In [ ]:
def export_pdf(html_path, report_id):
    pdf_path = OUTPUT_DIR / "pdf" / f"{report_id}.pdf"
    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        from weasyprint import HTML
        HTML(filename=str(html_path)).write_pdf(str(pdf_path))
        return pdf_path
    except Exception as exc:
        print(f"PDF export skipped: {exc}")
        return None

In [45]:
from app.report_service import generate_report

result = generate_report("2026-05-11", "2026-05-17", "html")

result

{'status': 'success',
 'report_id': 'weekly_sales_2026-05-11_2026-05-17',
 'markdown': 'C:\\Users\\Rayyanusa\\Downloads\\ai_auto_report_reference_solution\\Pertemuan 5\\outputs\\markdown\\weekly_sales_2026-05-11_2026-05-17.md',
 'html': 'C:\\Users\\Rayyanusa\\Downloads\\ai_auto_report_reference_solution\\Pertemuan 5\\outputs\\html\\weekly_sales_2026-05-11_2026-05-17.html',
 'pdf': None,
 'log': 'C:\\Users\\Rayyanusa\\Downloads\\ai_auto_report_reference_solution\\Pertemuan 5\\outputs\\logs\\weekly_sales_2026-05-11_2026-05-17.json',
 'alert_message': '[AUTO-REPORT ALERT]\n- HIGH | Revenue channel TikTok turun -37.2% vs periode sebelumnya\n  Cek: Cek campaign, diskon, stok, retur, dan perubahan traffic channel.\n- MEDIUM | Revenue channel Shopee turun -20.1% vs periode sebelumnya\n  Cek: Cek campaign, diskon, stok, retur, dan perubahan traffic channel.\nReport lengkap: C:\\Users\\Rayyanusa\\Downloads\\ai_auto_report_reference_solution\\Pertemuan 5\\outputs\\html\\weekly_sales_2026-05-11_2

In [46]:
from app.config import USE_MOCK_AI, OPENAI_API_KEY

print("USE_MOCK_AI:", USE_MOCK_AI)
print("API key ada:", bool(OPENAI_API_KEY))

USE_MOCK_AI: False
API key ada: True


In [47]:
from app.logging_utils import write_report_log

log_path = write_report_log(report_id, {
    "start_date": "2026-05-11",
    "end_date": "2026-05-17",
    "row_count": len(df),
    "alert_count": len(alert_df),
    "status": "success"
})

print(log_path)

C:\Users\Rayyanusa\Downloads\ai_auto_report_reference_solution\Pertemuan 5\outputs\logs\weekly_sales_2026-05-11_2026-05-17.json


In [48]:
from app.report_service import generate_report

result = generate_report(
    start_date="2026-05-11",
    end_date="2026-05-17",
    output_format="pdf"
)

result


-----

WeasyPrint could not import some external libraries. Please carefully follow the installation steps before reporting an issue:
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#installation
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#troubleshooting 

-----

PDF export skipped: cannot load library 'libgobject-2.0-0': error 0x7e.  Additionally, ctypes.util.find_library() did not manage to locate a library called 'libgobject-2.0-0'


{'status': 'success',
 'report_id': 'weekly_sales_2026-05-11_2026-05-17',
 'markdown': 'C:\\Users\\Rayyanusa\\Downloads\\ai_auto_report_reference_solution\\Pertemuan 5\\outputs\\markdown\\weekly_sales_2026-05-11_2026-05-17.md',
 'html': 'C:\\Users\\Rayyanusa\\Downloads\\ai_auto_report_reference_solution\\Pertemuan 5\\outputs\\html\\weekly_sales_2026-05-11_2026-05-17.html',
 'pdf': None,
 'log': 'C:\\Users\\Rayyanusa\\Downloads\\ai_auto_report_reference_solution\\Pertemuan 5\\outputs\\logs\\weekly_sales_2026-05-11_2026-05-17.json',
 'alert_message': '[AUTO-REPORT ALERT]\n- HIGH | Revenue channel TikTok turun -37.2% vs periode sebelumnya\n  Cek: Cek campaign, diskon, stok, retur, dan perubahan traffic channel.\n- MEDIUM | Revenue channel Shopee turun -20.1% vs periode sebelumnya\n  Cek: Cek campaign, diskon, stok, retur, dan perubahan traffic channel.\nReport lengkap: C:\\Users\\Rayyanusa\\Downloads\\ai_auto_report_reference_solution\\Pertemuan 5\\outputs\\html\\weekly_sales_2026-05-11_2